# Respostas às perguntas do case

Fonte: `ifood_case.refined.fct_taxi_trip` (fato unificado já aprovado nas
regras de qualidade) e os agregados `agg_trip_monthly` / `agg_trip_hourly`.

A primeira pergunta admite mais de uma interpretação plausível quanto ao
nível de agregação. Na segunda, a média de `passenger_count` por corrida,
agrupada pela hora do embarque, é a leitura direta; o fluxo médio de
passageiros por faixa horária é apresentado como análise complementar.

Em ambos os casos, o denominador, o escopo e a recomendação de uso são
explicitados para que as decisões analíticas não fiquem implícitas.

---
## Pergunta 1

> Qual a média de valor total (`total_amount`) recebido em um mês considerando
> todos os *yellow* táxis da frota?

**A ambiguidade:** "média de valor total recebido em um mês" pode significar

* **Leitura A — ticket médio:** média do `total_amount` *por corrida*, quebrada por mês.
  Responde "quanto vale, em média, uma corrida".
* **Leitura B — faturamento médio mensal:** soma do `total_amount` de cada mês e
  média dessas somas. Responde "quanto a frota fatura, em média, por mês".

As duas estão abaixo. A **Leitura B** é a que responde literalmente ao enunciado
("valor total recebido *em um mês*" = o que entrou no mês); a Leitura A é a
métrica que uma área de negócio normalmente quer acompanhar.

### 1.A · Ticket médio por corrida, por mês

In [0]:
%sql
SELECT
    reference_month                          AS mes,
    trip_count                               AS corridas,
    ROUND(avg_total_amount_per_trip, 2)      AS ticket_medio,
    ROUND(median_total_amount, 2)            AS mediana,
    ROUND(total_revenue, 2)                  AS receita_total
FROM ifood_case.refined.agg_trip_monthly
WHERE trip_type = 'yellow'
ORDER BY mes;

In [0]:
%sql
-- Mesmo número calculado direto do fato, sem passar pelo agregado.
-- Serve como prova de que o agregado está correto.
SELECT
    date_format(pickup_datetime, 'yyyy-MM')  AS mes,
    COUNT(*)                                 AS corridas,
    ROUND(AVG(total_amount), 2)              AS ticket_medio
FROM ifood_case.refined.fct_taxi_trip
WHERE trip_type = 'yellow'
GROUP BY ALL
ORDER BY mes;

### 1.B · Faturamento médio mensal da frota yellow

In [0]:
%sql
WITH por_mes AS (
    SELECT reference_month, total_revenue, trip_count
    FROM ifood_case.refined.agg_trip_monthly
    WHERE trip_type = 'yellow'
)
SELECT
    COUNT(*)                          AS meses_considerados,
    ROUND(SUM(total_revenue), 2)      AS receita_do_periodo,
    ROUND(AVG(total_revenue), 2)      AS receita_media_mensal,
    ROUND(AVG(trip_count), 0)         AS corridas_media_mensal
FROM por_mes;

### 1.C · Análise de sensibilidade

Quanto a decisão de mandar os `total_amount` negativos para a quarentena muda
a resposta? Se o impacto for irrelevante, a decisão é segura; se for grande,
ela precisa ser validada com a área de negócio antes de virar número oficial.

In [0]:
%sql
WITH todas AS (
    SELECT total_amount, 'aprovadas (fato)' AS cenario
    FROM ifood_case.refined.fct_taxi_trip WHERE trip_type = 'yellow'
    UNION ALL
    SELECT total_amount, 'aprovadas + negativos'
    FROM ifood_case.refined.fct_taxi_trip WHERE trip_type = 'yellow'
    UNION ALL
    SELECT total_amount, 'aprovadas + negativos'
    FROM ifood_case.refined.rej_taxi_trip
    WHERE trip_type = 'yellow'
      AND array_contains(_rejection_reasons, 'total_amount_negativo')
      AND size(_rejection_reasons) = 1
)
SELECT
    cenario,
    COUNT(*)                        AS corridas,
    ROUND(AVG(total_amount), 4)     AS ticket_medio,
    ROUND(SUM(total_amount), 2)     AS receita
FROM todas
GROUP BY cenario
ORDER BY cenario;

---
## Pergunta 2

> Qual a média de passageiros (`passenger_count`) por cada hora do dia que
> pegaram táxi no mês de maio considerando todos os táxis da frota?

**Escopo — "todos os táxis da frota":** yellow e green. As demais bases da
TLC (FHV e High Volume FHV, que incluem Uber/Lyft) pertencem a categorias de serviço distintas dos táxis de medalhão e não disponibilizam passenger_count.

**Interpretação da métrica:**

- **Leitura A — ocupação média:** média de `passenger_count` por corrida,
  agrupada pela hora do embarque. Responde quantas pessoas viajaram juntas,
  em média, em cada horário.
- **Leitura B — volume médio de passageiros:** total de passageiros
  transportados em cada faixa horária dividido pelos 31 dias de maio.
  Representa o fluxo médio de passageiros naquela hora em um dia típico.

A **Leitura A** corresponde diretamente à média da coluna `passenger_count`
solicitada no enunciado e, portanto, é apresentada como resposta principal.
A Leitura B é uma análise complementar, útil para compreender a demanda
horária e apoiar discussões de capacidade operacional.

**Filtro aplicado:** registros com `passenger_count` nulo ou igual a zero
foram excluídos apenas desta análise, sem serem removidos da base. Valores
nulos não participariam do cálculo de `AVG`; já os zeros reduziriam a média
como se representassem corridas sem passageiros. Por isso, a versão curada
considera somente valores positivos, e a diferença em relação ao cálculo
literal é apresentada na análise de sensibilidade.

### 2.A · Ocupação média por hora do dia (maio/2023, frota completa)

In [0]:
%sql
SELECT
    LPAD(pickup_hour, 2, '0')                                  AS hora,
    SUM(trips_with_passenger_count)                            AS corridas,
    ROUND(SUM(total_passengers) / SUM(trips_with_passenger_count), 4) AS media_passageiros_por_corrida
FROM ifood_case.refined.agg_trip_hourly
WHERE pickup_year = '2023' AND pickup_month = '05'
GROUP BY pickup_hour
ORDER BY pickup_hour;

In [0]:
%sql
-- Conferência direto no fato.
SELECT
    LPAD(hour(pickup_datetime), 2, '0')     AS hora,
    COUNT(*)                                AS corridas,
    ROUND(AVG(passenger_count), 4)          AS media_passageiros_por_corrida
FROM ifood_case.refined.fct_taxi_trip
WHERE pickup_datetime >= '2023-05-01'
  AND pickup_datetime <  '2023-06-01'
  AND passenger_count > 0
GROUP BY ALL
ORDER BY hora;

### 2.B · Passageiros por hora num dia típico de maio

In [0]:
%sql
SELECT
    LPAD(pickup_hour, 2, '0')                            AS hora,
    SUM(total_passengers)                                AS passageiros_no_mes,
    MAX(distinct_days)                                   AS dias_com_dado,
    ROUND(SUM(total_passengers) / MAX(distinct_days), 1) AS passageiros_por_hora_dia_tipico
FROM ifood_case.refined.agg_trip_hourly
WHERE pickup_year = '2023' AND pickup_month = '05'
GROUP BY pickup_hour
ORDER BY pickup_hour;

### 2.C · Abertura por tipo de táxi

Yellow e green têm perfis de ocupação diferentes; vale mostrar.

In [0]:
%sql
SELECT
    LPAD(pickup_hour, 2, '0') AS hora,
    ROUND(MAX(CASE WHEN trip_type = 'yellow' THEN avg_passenger_count END), 4) AS yellow,
    ROUND(MAX(CASE WHEN trip_type = 'green'  THEN avg_passenger_count END), 4) AS green
FROM ifood_case.refined.agg_trip_hourly
WHERE pickup_year = '2023' AND pickup_month = '05'
GROUP BY pickup_hour
ORDER BY pickup_hour;

### 2.D · Visualização

In [0]:
import matplotlib.pyplot as plt

df = spark.sql("""
    SELECT pickup_hour,
           SUM(total_passengers) / SUM(trips_with_passenger_count) AS ocupacao_media,
           SUM(trips_with_passenger_count)                          AS corridas
    FROM ifood_case.refined.agg_trip_hourly
    WHERE pickup_year = '2023' AND pickup_month = '05'
    GROUP BY pickup_hour ORDER BY pickup_hour
""").toPandas()

fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.bar(df['pickup_hour'], df['corridas'], alpha=0.25, label='Corridas')
ax1.set_xlabel('Hora do embarque')
ax1.set_ylabel('Corridas')
ax1.set_xticks(range(24))

ax2 = ax1.twinx()
ax2.plot(df['pickup_hour'], df['ocupacao_media'], marker='o', linewidth=2, label='Ocupação média')
ax2.set_ylabel('Passageiros por corrida')

plt.title('Maio/2023 — volume de corridas e ocupação média por hora (yellow + green)')
fig.tight_layout()
plt.show()

---
## Como ler estes números

1. **Ticket médio é estável entre os meses, faturamento não.** A variação mensal
   da receita vem principalmente de volume de corridas, não de preço por corrida.
2. **A média do `total_amount` fica acima da mediana**, sinal de distribuição com cauda
   longa à direita. As colunas do case não permitem identificar o que compõe essa cauda:
   seria preciso `trip_distance` e `RatecodeID`. Para acompanhamento operacional, a
   mediana descreve melhor a corrida típica.
3. **A ocupação média por hora varia pouco**, entre 1,26 e 1,46 passageiro por corrida.
   O que varia de verdade é o *volume*. Por isso a Leitura B importa tanto quanto a
   Leitura A para qualquer decisão de operação. As colunas disponíveis não permitem
   atribuir causa a esse padrão: seria preciso localização, dia da semana e tipo de corrida.

> Os números desta execução estão consolidados em `analysis/RESULTADOS.md`,
> junto com as SQLs de reprodução.